In [31]:
import pandas as pd
import numpy as np
from numpy.ma.core import remainder
from sklearn.metrics._plot import confusion_matrix
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.neighbors import KNeighborsClassifier
from tqdm import tqdm
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import VotingClassifier,VotingRegressor,BaggingClassifier
from sklearn.preprocessing import StandardScaler,OneHotEncoder,LabelEncoder
from sklearn.metrics import classification_report, f1_score, accuracy_score, log_loss, ConfusionMatrixDisplay,r2_score
from sklearn.model_selection import train_test_split
from sklearn.compose import make_column_selector,make_column_transformer
from sklearn.linear_model import  LogisticRegression

In [32]:
df=pd.read_csv("../Datasets/Sonar.csv")

In [33]:
df.columns

Index(['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11',
       'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21',
       'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'V29', 'V30', 'V31',
       'V32', 'V33', 'V34', 'V35', 'V36', 'V37', 'V38', 'V39', 'V40', 'V41',
       'V42', 'V43', 'V44', 'V45', 'V46', 'V47', 'V48', 'V49', 'V50', 'V51',
       'V52', 'V53', 'V54', 'V55', 'V56', 'V57', 'V58', 'V59', 'V60', 'Class'],
      dtype='str')

In [34]:
le=LabelEncoder()
X=df.drop(columns=["Class"])
Y=le.fit_transform(df["Class"])

In [35]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.3,random_state=25,stratify=Y)

In [36]:
ddc=DecisionTreeClassifier()
nb=GaussianNB()
knn=KNeighborsClassifier()
lr=LogisticRegression()
bag=BaggingClassifier(estimator=nb,n_estimators=10,random_state=25)
bag.fit(X_train,Y_train)
y_pred=bag.predict(X_test)
print(classification_report(Y_test,y_pred))

              precision    recall  f1-score   support

           0       0.62      0.74      0.68        34
           1       0.61      0.48      0.54        29

    accuracy                           0.62        63
   macro avg       0.62      0.61      0.61        63
weighted avg       0.62      0.62      0.61        63



In [37]:
bag=BaggingClassifier(estimator=knn,n_estimators=10,random_state=25)
bag.fit(X_train,Y_train)
y_pred=bag.predict(X_test)
print(classification_report(Y_test,y_pred,target_names=le.classes_))


              precision    recall  f1-score   support

           M       0.74      0.85      0.79        34
           R       0.79      0.66      0.72        29

    accuracy                           0.76        63
   macro avg       0.77      0.75      0.76        63
weighted avg       0.77      0.76      0.76        63



In [38]:
bag=BaggingClassifier(estimator=ddc,n_estimators=10,random_state=25)
bag.fit(X_train,Y_train)
y_pred=bag.predict(X_test)
print(classification_report(Y_test,y_pred,target_names=le.classes_))

              precision    recall  f1-score   support

           M       0.67      0.94      0.78        34
           R       0.87      0.45      0.59        29

    accuracy                           0.71        63
   macro avg       0.77      0.69      0.69        63
weighted avg       0.76      0.71      0.69        63



In [39]:
bag=BaggingClassifier(estimator=lr,n_estimators=10,random_state=25)
bag.fit(X_train,Y_train)
y_pred=bag.predict(X_test)
print(classification_report(Y_test,y_pred,target_names=le.classes_))

              precision    recall  f1-score   support

           M       0.75      0.88      0.81        34
           R       0.83      0.66      0.73        29

    accuracy                           0.78        63
   macro avg       0.79      0.77      0.77        63
weighted avg       0.79      0.78      0.77        63



In [40]:
# est lis
estList=[lr,ddc,nb,knn]
no_of_est=[10,15,25,50]
scores=list()

for est in tqdm(estList):
    for number in no_of_est:
        bagging=BaggingClassifier(estimator=est,n_estimators=number,random_state=25)
        bagging.fit(X_train,Y_train)
        y_pred=bagging.predict(X_test)
        y_prob=bagging.predict_proba(X_test)
        scores.append([est,number,log_loss(Y_test,y_prob)])

score_df=pd.DataFrame(scores,columns=["estimator","number","loss_score"]).sort_values("loss_score",ascending=True)
print(score_df)

100%|██████████| 4/4 [00:01<00:00,  2.31it/s]

                   estimator  number  loss_score
12    KNeighborsClassifier()      10    0.422282
13    KNeighborsClassifier()      15    0.431490
14    KNeighborsClassifier()      25    0.432707
15    KNeighborsClassifier()      50    0.447037
6   DecisionTreeClassifier()      25    0.475750
4   DecisionTreeClassifier()      10    0.477146
7   DecisionTreeClassifier()      50    0.490503
5   DecisionTreeClassifier()      15    0.502120
1       LogisticRegression()      15    0.534413
0       LogisticRegression()      10    0.537041
3       LogisticRegression()      50    0.537758
2       LogisticRegression()      25    0.537879
8               GaussianNB()      10    2.450082
9               GaussianNB()      15    2.501690
10              GaussianNB()      25    2.510042
11              GaussianNB()      50    2.528499


In [41]:
df_hr=pd.read_csv("../Datasets/HR_comma_sep.csv")
df_hr.columns

Index(['satisfaction_level', 'last_evaluation', 'number_project',
       'average_montly_hours', 'time_spend_company', 'Work_accident', 'left',
       'promotion_last_5years', 'Department', 'salary'],
      dtype='str')

In [42]:
X=df_hr.drop(columns=["left"])
Y=df_hr["left"]

In [43]:
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.3,random_state=25,stratify=Y)

In [44]:
ohe=OneHotEncoder(drop="first",sparse_output=False).set_output(transform="pandas")
trms=make_column_transformer((ohe,make_column_selector(dtype_include="object")),remainder="passthrough",verbose_feature_names_out=False)
X_train_trms=trms.fit_transform(X_train)
X_test_trms=trms.transform(X_test)

In [49]:
nb=GaussianNB()
knn=KNeighborsClassifier()
ddc=DecisionTreeClassifier()
lr=LogisticRegression()

In [46]:
bag=BaggingClassifier(estimator=nb,n_estimators=10,random_state=25)
bag.fit(X_train_trms,Y_train)
y_pred=bag.predict(X_test_trms)
print(classification_report(Y_test,y_pred,target_names=le.classes_))

              precision    recall  f1-score   support

           M       0.90      0.70      0.79      3429
           R       0.44      0.76      0.56      1070

    accuracy                           0.71      4499
   macro avg       0.67      0.73      0.67      4499
weighted avg       0.79      0.71      0.73      4499



In [47]:
bag=BaggingClassifier(estimator=knn,n_estimators=10,random_state=25)
bag.fit(X_train_trms,Y_train)
y_pred=bag.predict(X_test_trms)
print(classification_report(Y_test,y_pred,target_names=le.classes_))

              precision    recall  f1-score   support

           M       0.98      0.94      0.96      3429
           R       0.83      0.93      0.88      1070

    accuracy                           0.94      4499
   macro avg       0.91      0.94      0.92      4499
weighted avg       0.94      0.94      0.94      4499



In [48]:
bag=BaggingClassifier(estimator=ddc,n_estimators=10,random_state=25)
bag.fit(X_train_trms,Y_train)
y_pred=bag.predict(X_test_trms)
print(classification_report(Y_test,y_pred,target_names=le.classes_))

              precision    recall  f1-score   support

           M       0.99      1.00      0.99      3429
           R       0.99      0.98      0.98      1070

    accuracy                           0.99      4499
   macro avg       0.99      0.99      0.99      4499
weighted avg       0.99      0.99      0.99      4499



In [52]:
# lr,De,knn,nb
# 10,15,25,50

from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import log_loss
import pandas as pd
from tqdm import tqdm

est_lists = [lr, ddc, nb, knn]
no_of_lists = [10, 15, 25, 50]

scores = []

for est in tqdm(est_lists):
    for number in no_of_lists:
        bagging = BaggingClassifier(
            estimator=est,
            n_estimators=number,
            random_state=25
        )

        bagging.fit(X_train_trms, Y_train)

        y_prob = bagging.predict_proba(X_test_trms)

        scores.append([
            est.__class__.__name__,   # Estimator name
            number,
            log_loss(Y_test, y_prob)
        ])

score_df = (
    pd.DataFrame(
        scores,
        columns=["Estimator", "n_estimators", "Log Loss"]
    )
    .sort_values("Log Loss")
)

print(score_df)

  0%|          | 0/4 [00:00<?, ?it/s]c:\Users\UserE\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\UserE\miniconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/mo

                 Estimator  n_estimators  Log Loss
7   DecisionTreeClassifier            50  0.128360
6   DecisionTreeClassifier            25  0.149958
5   DecisionTreeClassifier            15  0.158270
4   DecisionTreeClassifier            10  0.173862
15    KNeighborsClassifier            50  0.417508
3       LogisticRegression            50  0.429809
2       LogisticRegression            25  0.430052
1       LogisticRegression            15  0.430141
0       LogisticRegression            10  0.430273
14    KNeighborsClassifier            25  0.452316
13    KNeighborsClassifier            15  0.465798
12    KNeighborsClassifier            10  0.480350
11              GaussianNB            50  0.714871
10              GaussianNB            25  0.715199
9               GaussianNB            15  0.724410
8               GaussianNB            10  0.740191


In [55]:
depth=[None,3,5,7]
scores=[]
for est in tqdm(depth):
     ddc=DecisionTreeClassifier(random_state=25,max_depth=est)
     bagging=BaggingClassifier(estimator=ddc,n_estimators=50,random_state=25)
     bagging.fit(X_train_trms, Y_train)
     y_pred=bagging.predict(X_test_trms)
     y_prob=bagging.predict_proba(X_test_trms)
     scores.append([est,log_loss(Y_test,y_prob)])

score_df=pd.DataFrame(scores,columns=["E","Log Loss"]).sort_values("Log Loss", ascending=True)
print(score_df)

100%|██████████| 4/4 [00:02<00:00,  1.76it/s]

     E  Log Loss
3  7.0  0.079241
2  5.0  0.088876
0  NaN  0.128360
1  3.0  0.138152
